In [1]:
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)

TABLES = Path("../pipeline_tables")   # the stage summary tables this repo ships


In [2]:
# What came out of internal QC, what the fixing stage recovered, and the reference typing
qc = pd.read_csv(TABLES / "internal_qc_summary.tsv", sep="\t")
decontam = pd.read_csv(TABLES / "decontam_typing_resolved.tsv", sep="\t")
meta = pd.read_csv(TABLES / "KASPAH_collection_nr_170719.tsv", sep="\t")

len(qc), len(decontam), len(meta)

(316, 6, 391)

In [3]:
qc["internal_qc_decision"].value_counts()

internal_qc_decision
PASS                               288
FAIL_VARIANT                        10
REDO_DECONTAM_KLEBSIELLA             4
PASS_PROVISIONAL_CHECK_EXTERNAL      4
PASS_BORDERLINE_MAPPING              3
REDO_ASSEMBLY_META                   3
PASS_FALSE_POSITIVE_CONTAM           2
REDO_DECONTAM_ECOLI                  2
Name: count, dtype: int64

In [4]:
# Survivors = everything internal QC passed, plus what 06_asm_fixing recovered.
# Discarded there: INF096, INF294 (01_autocycler/failed), INF262_v2 (02_decontam).
RECOVERED = ["INF205",                                                    # 01_autocycler
             "INF186", "INF216", "KSB1_2I", "KSB2_4A", "KSB2_5D_v2"]      # 02_decontam

survivors = qc[qc["internal_qc_decision"].str.startswith("PASS")
               | qc["sample"].isin(RECOVERED)].copy()

len(survivors)

303

In [5]:
# The 5 decontaminated strains were re-typed after reassembly - their internal_qc typing
# describes the mixed assembly, so it has to be replaced. INF205 was only re-assembled,
# not re-typed; its ONT/Illumina typing was already concordant, so it carries over.
KAPTIVE = {
    "enterobacterales__species__species": "kbt_ont_species",
    "klebsiella_pneumo_complex__mlst__ST": "kbt_ont_ST",
    "klebsiella_pneumo_complex__kaptive__K_locus": "kbt_ont_K_locus",
    "klebsiella_pneumo_complex__kaptive__K_locus_confidence": "kbt_ont_K_locus_confidence",
}
retyped = decontam.rename(columns=KAPTIVE).set_index("strain")[list(KAPTIVE.values())]

survivors = survivors.set_index("sample")
survivors.update(retyped)
survivors = survivors.reset_index()

survivors[survivors["sample"].isin(RECOVERED)][
    ["sample", "kbt_ont_species", "kbt_ont_ST", "kbt_ont_K_locus", "internal_qc_decision"]]

,sample,kbt_ont_species,kbt_ont_ST,kbt_ont_K_locus,internal_qc_decision
120,INF186,Klebsiella quasipneumoniae subsp. similipneumo...,ST480,KL52,REDO_DECONTAM_KLEBSIELLA
134,INF205,Klebsiella pneumoniae,ST3049,KL55,REDO_ASSEMBLY_META
140,INF216,Klebsiella variicola subsp. variicola,ST3079,KL53,REDO_DECONTAM_KLEBSIELLA
240,KSB1_2I,Klebsiella variicola subsp. variicola,ST2993,KL151,REDO_DECONTAM_ECOLI
287,KSB2_4A,Klebsiella quasipneumoniae subsp. similipneumo...,ST1191,KL13,REDO_DECONTAM_ECOLI
290,KSB2_5D_v2,Klebsiella variicola subsp. variicola,ST6940,KL53,REDO_DECONTAM_KLEBSIELLA


In [6]:
# Metadata is keyed on the isolate, assemblies on the library: INF255.1_v2 -> INF255.
# .1/.2 = morphological variants of one isolate, _v2 = resequenced in this project.
def isolate_of(sample):
    return re.sub(r"\.\d+$", "", re.sub(r"_v\d+$", "", sample))


survivors["isolate"] = survivors["sample"].map(isolate_of)

# nothing may fall outside the reference collection
set(survivors["isolate"]) - set(meta["sample ID"])

set()

In [7]:
m = meta.set_index("sample ID")
survivors["ext_species"] = survivors["isolate"].map(m["species"])
survivors["ext_ST"] = survivors["isolate"].map(m["ST"])
survivors["ext_K_locus"] = survivors["isolate"].map(m["K_locus"])
survivors["ext_K_locus_confidence"] = survivors["isolate"].map(m["K_locus_confidence"])
survivors["ext_K_locus_identity"] = survivors["isolate"].map(m["K_locus_identity"])
survivors["ext_K_locus_problems"] = survivors["isolate"].map(m["K_locus_problems"])
survivors["ext_K_locus_missing_genes"] = survivors["isolate"].map(m["K_locus_missing_genes"])
survivors["ext_contig_count"] = survivors["isolate"].map(m["contig_count"])

In [8]:
# The Illumina assembly of each library was typed in the same Kleborate run as the ONT one,
# under <sample>_illumina. Both sides of the internal comparison come from here.
SPECIES = "enterobacterales__species__species"
K_LOCUS = "klebsiella_pneumo_complex__kaptive__K_locus"
LOCI = ["gapA", "infB", "mdh", "pgi", "phoE", "rpoB", "tonB"]

kleborate = pd.read_csv(
    "../04_verification/kleborate/results/klebsiella_pneumo_complex_output.txt", sep="\t")
kleborate["strain"] = kleborate["strain"].str.replace("_ont$", "", regex=True)
kleborate = kleborate.set_index("strain")

# the decontaminated strains are re-typed here too, replacing their pre-fix ONT row
typing = pd.concat([kleborate, decontam.set_index("strain")])
typing = typing[~typing.index.duplicated(keep="last")]

assert survivors["sample"].isin(typing.index).all()
assert survivors["sample"].add("_illumina").isin(typing.index).all()

In [9]:
alleles = typing[["klebsiella_pneumo_complex__mlst__" + locus for locus in LOCI]]
alleles.columns = LOCI
alleles = alleles.astype(str)
alleles.head()

,gapA,infB,mdh,pgi,phoE,rpoB,tonB
strain,,,,,,,
INF001.1_illumina,16,45,223,132,47,17,67
INF001.1,16,45,223,132,47*,17,67
INF001.2_illumina,16,45,223,132,47,17,67
INF001.2,7,1,65,1,9,4,18
INF002_illumina,12,1,1,2,5,1,36


In [10]:
# ST names are not comparable. "ST1234-2LV" is a nearest neighbour, not a call, and the
# reference was typed against an older MLST database, so a profile that had no ST then may
# have one now - the same seven alleles can carry two different names. Compare the alleles.
#
# Kleborate marks an inexact allele "1*" and a missing one "-". The number is the identity;
# drop the marker and compare on that. A locus missing on either side counts as a difference.
MISSING = {"-", "", "nan", "NaN", "0"}


def allele(value):
    return re.sub(r"[*^?]", "", str(value)).strip()


def allele_differences(ours, theirs):
    """Loci on which two seven-locus profiles disagree."""
    diff = []
    for locus in LOCI:
        a, b = allele(ours[locus]), allele(theirs[locus])
        if a in MISSING or b in MISSING or a != b:
            diff.append(f"{locus} {theirs[locus]} -> {ours[locus]}")
    return diff

In [11]:
# --- against the reference collection --------------------------------------------------
survivors["ext_species_match"] = survivors["kbt_ont_species"] == survivors["ext_species"]
survivors["ext_K_locus_match"] = survivors["kbt_ont_K_locus"] == survivors["ext_K_locus"]
survivors["ext_mlst_distance"] = [
    len(allele_differences(alleles.loc[sample], m.loc[isolate_of(sample), LOCI]))
    for sample in survivors["sample"]]

survivors["ext_mlst_distance"].value_counts().sort_index()

ext_mlst_distance
0    287
1      4
3      1
4      3
5      3
6      1
7      4
Name: count, dtype: int64

In [12]:
# Distances split cleanly: 0-1 on one side, 3-7 on the other, nothing at 2. The threshold
# sits in a real gap rather than being drawn through a continuum.
MLST_MAX_DISTANCE = 2

survivors["ext_mlst_match"] = survivors["ext_mlst_distance"] <= MLST_MAX_DISTANCE
survivors["matches_reference"] = (survivors["ext_species_match"]
                                  & survivors["ext_K_locus_match"]
                                  & survivors["ext_mlst_match"])

survivors["matches_reference"].value_counts()

matches_reference
True     287
False     16
Name: count, dtype: int64

In [13]:
# --- ONT assembly vs Illumina assembly of the same library ------------------------------
illumina = survivors["sample"] + "_illumina"

survivors["int_species_match"] = (survivors["kbt_ont_species"].values
                                  == typing.loc[illumina, SPECIES].values)
survivors["int_K_locus_match"] = (survivors["kbt_ont_K_locus"].values
                                  == typing.loc[illumina, K_LOCUS].values)
survivors["int_mlst_distance"] = [
    len(allele_differences(alleles.loc[sample], alleles.loc[sample + "_illumina"]))
    for sample in survivors["sample"]]
survivors["int_mlst_match"] = survivors["int_mlst_distance"] <= MLST_MAX_DISTANCE

survivors["internally_concordant"] = (survivors["int_species_match"]
                                      & survivors["int_K_locus_match"]
                                      & survivors["int_mlst_match"])

survivors["internally_concordant"].value_counts()

internally_concordant
True     302
False      1
Name: count, dtype: int64

In [14]:
# Two questions, two answers
pd.crosstab(survivors["internally_concordant"], survivors["matches_reference"])

matches_reference,False,True
internally_concordant,,
False,0,1
True,16,286


In [15]:
COMPARE = ["sample", "kbt_ont_species", "kbt_ont_ST", "kbt_ont_K_locus",
           "ext_species", "ext_ST", "ext_K_locus",
           "ext_species_match", "ext_K_locus_match", "ext_mlst_distance"]

# --- internally fine, does not match the reference --------------------------------------
mismatched = survivors[survivors["internally_concordant"] & ~survivors["matches_reference"]]
mismatched[COMPARE]

,sample,kbt_ont_species,kbt_ont_ST,kbt_ont_K_locus,ext_species,ext_ST,ext_K_locus,ext_species_match,ext_K_locus_match,ext_mlst_distance
22,INF033,Klebsiella pneumoniae,ST27,KL22,Klebsiella pneumoniae,ST27,KL37,True,False,0
23,INF034_v2,Klebsiella pneumoniae,ST15-1LV,KL24,Klebsiella quasipneumoniae subsp. similipneumo...,ST3097-1LV,KL7,False,False,7
109,INF167_v2,Klebsiella pneumoniae,ST17,KL25,Klebsiella pneumoniae,ST2370,KL15,True,False,4
121,INF187_v2,Klebsiella pneumoniae,ST23,KL1,Klebsiella quasipneumoniae subsp. similipneumo...,ST480,KL52,False,False,7
131,INF202_v2,Klebsiella pneumoniae,ST292,KL143,Klebsiella pneumoniae,ST3052,KL112,True,False,5
148,INF226_v2,Klebsiella pneumoniae,ST336,KL25,Klebsiella pneumoniae,ST3068,KL119,True,False,4
149,INF227_v2,Klebsiella pneumoniae,ST86,KL2,Klebsiella pneumoniae,ST792,KL25,True,False,6
150,INF229_v2,Klebsiella pneumoniae,ST3068,KL119,Klebsiella pneumoniae,ST221,KL50,True,False,5
162,INF251_v2,Klebsiella pneumoniae,ST2623,KL52,Klebsiella pneumoniae,ST485,KL28,True,False,3
167,INF255.1_v2,Klebsiella pneumoniae,ST29,KL30,Klebsiella variicola subsp. variicola,ST681,KL10,False,False,7


In [16]:
# These split by how much disagrees. At MLST distance >= 3 the species, K locus and the
# whole seven-locus profile go together - that is a different strain, so the reference row
# describes something other than this genome. At distance 0-1 only the K locus differs.
mismatched.groupby(mismatched["ext_mlst_distance"] >= 3)[
    ["ext_species_match", "ext_K_locus_match"]].sum()

,ext_species_match,ext_K_locus_match
ext_mlst_distance,,
False,4,0
True,8,0


In [17]:
# For a whole-profile mismatch, ask which reference isolate the genome *does* look like.
# A hit means the reference row is attached to the wrong isolate.
def best_external_match(row):
    hits = meta[(meta["species"] == row["kbt_ont_species"])
                & (meta["K_locus"] == row["kbt_ont_K_locus"])
                & (meta["ST"].str.replace(r"-\d+LV$", "", regex=True)
                   == re.sub(r"-\d+LV$", "", str(row["kbt_ont_ST"])))]
    return ",".join(hits["sample ID"])


survivors["best_external_match"] = survivors.apply(
    lambda row: "" if row["matches_reference"] else best_external_match(row), axis=1)

different_strain = mismatched[mismatched["ext_mlst_distance"] >= 3]
survivors.loc[different_strain.index, ["sample", "kbt_ont_species", "kbt_ont_ST",
                                       "kbt_ont_K_locus", "ext_mlst_distance",
                                       "best_external_match"]]

,sample,kbt_ont_species,kbt_ont_ST,kbt_ont_K_locus,ext_mlst_distance,best_external_match
23,INF034_v2,Klebsiella pneumoniae,ST15-1LV,KL24,7,"KSB1_7C,INF061,KSB1_5B"
109,INF167_v2,Klebsiella pneumoniae,ST17,KL25,4,"INF300,INF359,KSB1_9J"
121,INF187_v2,Klebsiella pneumoniae,ST23,KL1,7,"KSB1_10A,INF079"
131,INF202_v2,Klebsiella pneumoniae,ST292,KL143,5,INF200
148,INF226_v2,Klebsiella pneumoniae,ST336,KL25,4,INF224
149,INF227_v2,Klebsiella pneumoniae,ST86,KL2,6,"INF191,INF225,INF254,INF331"
150,INF229_v2,Klebsiella pneumoniae,ST3068,KL119,5,INF226
162,INF251_v2,Klebsiella pneumoniae,ST2623,KL52,3,
167,INF255.1_v2,Klebsiella pneumoniae,ST29,KL30,7,"INF261,INF282,INF289,INF314,INF246,INF332,INF1..."
169,INF257.1_v2,Klebsiella pneumoniae,ST3074,KL13,4,INF258


In [18]:
# Every one of them is a _v2 library and every one is ONT/Illumina concordant, so the
# genome is not in doubt - two independent read sets off the same tube agree with each
# other and disagree with the reference row. Eight of them type as another isolate in the
# collection outright. Kept and flagged; the metadata is what needs revisiting.

In [19]:
# --- the K locus only, at MLST distance 0-1 ---------------------------------------------
K_locus_only = mismatched[mismatched["ext_mlst_distance"] < 3]
K_locus_only[["sample", "kbt_ont_K_locus", "kbt_ont_K_locus_confidence", "ext_mlst_distance",
              "ext_K_locus", "ext_K_locus_confidence", "ext_K_locus_identity",
              "ext_K_locus_problems", "ext_contig_count", "ext_K_locus_missing_genes"]].T

,22,177,202,245
sample,INF033,INF267,INF311,KSB1_3E
kbt_ont_K_locus,KL22,KL64,KL144,KL22
kbt_ont_K_locus_confidence,Typeable,Typeable,Untypeable,Typeable
ext_mlst_distance,0,0,0,1
ext_K_locus,KL37,KL107,KL107,KL37
ext_K_locus_confidence,Good,NaN,NaN,Good
ext_K_locus_identity,"99,95%","90,24%","83,17%","99,96%"
ext_K_locus_problems,+,?-+*,?-+*,+
ext_contig_count,207,104,220,289
ext_K_locus_missing_genes,NaN,"KL107_05_wzb,KL107_06_wzc,KL107_07_wbaP,KL107_...","KL107_05_wzb,KL107_06_wzc,KL107_07_wbaP,KL107_...",NaN


In [20]:
# All four reference calls come off a fragmented short-read assembly (104-289 contigs), and
# they fail in two different ways. Split them on whether the reference call is usable at
# all: a locus reported with missing expected genes, or below 95% identity, is not a call.
def reference_K_locus_usable(row):
    identity = float(str(row["ext_K_locus_identity"]).replace(",", ".").rstrip("%"))
    return identity >= 95 and pd.isna(row["ext_K_locus_missing_genes"])


K_locus_only.apply(reference_K_locus_usable, axis=1)

22      True
177    False
202    False
245     True
dtype: bool

In [21]:
# INF267, INF311   reference KL107 at 83-90% identity with six missing expected genes
#                  (wzb, wzc, wbaP among them) - not a call. The closed assemblies
#                  supersede it (KL64; KL144, itself Untypeable).
# INF033, KSB1_3E  reference KL37 at ~99.95% identity vs KL22 here. KL22 and KL37 are near
#                  identical loci and both carry wzi90; on a 207- and a 289-contig assembly
#                  the locus is split across contigs and either can win.
#
# Both kept - the reference K locus is the unreliable side either way.

In [22]:
# --- ONT and Illumina disagree ----------------------------------------------------------
survivors[~survivors["internally_concordant"]][
    ["sample", "kbt_ont_species", "kbt_ont_ST", "int_species_match", "int_K_locus_match",
     "int_mlst_distance", "matches_reference"]]

,sample,kbt_ont_species,kbt_ont_ST,int_species_match,int_K_locus_match,int_mlst_distance,matches_reference
290,KSB2_5D_v2,Klebsiella variicola subsp. variicola,ST6940,False,True,7,True


In [23]:
# KSB2_5D_v2's ONT reads were a K. variicola / K. pneumoniae mixture; 06_asm_fixing kept the
# variicola, which reproduces the reference on species, K locus and all seven MLST loci
# (tonB 74* -> 945, resolving the reference's ST1217-3LV to ST6940). The Illumina library is
# the other half of that mixture - K. pneumoniae ST35, sharing only the KL53 capsule call,
# which 06_asm_fixing/02_decontam already noted.
#
# So there is no hybrid assembly to build here: the two read sets are different organisms,
# and polishing the variicola assembly with pneumoniae reads would corrupt it. Dropped.
typing.loc[["KSB2_5D_v2", "KSB2_5D_v2_illumina"],
           [SPECIES, "klebsiella_pneumo_complex__mlst__ST", K_LOCUS]]

,enterobacterales__species__species,klebsiella_pneumo_complex__mlst__ST,klebsiella_pneumo_complex__kaptive__K_locus
strain,,,
KSB2_5D_v2,Klebsiella variicola subsp. variicola,ST6940,KL53
KSB2_5D_v2_illumina,Klebsiella pneumoniae,ST35-2LV,KL53


In [24]:
# --- MLST differences that are not disagreements ----------------------------------------
# Within the threshold but not identical: each differs at one locus, and at a locus the
# reference had called inexactly (starred). The closed assembly resolves it.
near = survivors[survivors["ext_mlst_distance"].between(1, MLST_MAX_DISTANCE)]
for sample in near["sample"]:
    print(f"{sample:14s} {m.loc[isolate_of(sample), 'ST']:12s} "
          f"{allele_differences(alleles.loc[sample], m.loc[isolate_of(sample), LOCI])}")

KSB1_3E        ST10-1LV     ['rpoB 1* -> 378']
KSB1_4D        ST277-1LV    ['phoE 1* -> 648']
KSB1_4G        ST2981-3LV   ['tonB 52* -> 943']
KSB2_5D_v2     ST1217-3LV   ['tonB 74* -> 945']


In [25]:
# And identical on all seven loci while carrying a different ST name: the same strain under
# an ST the reference's MLST database did not have.
renamed = survivors[(survivors["ext_mlst_distance"] == 0)
                    & (survivors["kbt_ont_ST"] != survivors["ext_ST"])]
renamed[["sample", "ext_ST", "kbt_ont_ST", "ext_mlst_distance"]]

,sample,ext_ST,kbt_ont_ST,ext_mlst_distance
0,INF001.1,ST3090,ST3090-1LV,0
6,INF007,ST1449,ST1449-1LV,0
24,INF035_v2,ST3097-1LV,ST5872,0
50,INF072,ST133,ST133-1LV,0
103,INF155,ST882,ST882-1LV,0
130,INF200,ST292,ST292-1LV,0
227,INF360,ST323,ST323-1LV,0
230,KSB1_1D,ST45,ST45-1LV,0
237,KSB1_2E,ST993-1LV,ST6935,0
250,KSB1_4A,ST2800-1LV,ST5142,0


In [26]:
# Clear the four that internal QC passed provisionally pending external typing
PROVISIONAL = ["INF195", "KSB1_2F", "KSB1_3I", "KSB1_6D_v2"]
survivors[survivors["sample"].isin(PROVISIONAL)][COMPARE]

,sample,kbt_ont_species,kbt_ont_ST,kbt_ont_K_locus,ext_species,ext_ST,ext_K_locus,ext_species_match,ext_K_locus_match,ext_mlst_distance
125,INF195,Klebsiella quasipneumoniae subsp. similipneumo...,ST414,KL123,Klebsiella quasipneumoniae subsp. similipneumo...,ST414,KL123,True,True,0
238,KSB1_2F,Klebsiella variicola subsp. variicola,ST2594,KL134,Klebsiella variicola subsp. variicola,ST2594,KL134,True,True,0
248,KSB1_3I,Klebsiella quasipneumoniae subsp. similipneumo...,ST1548,KL170,Klebsiella quasipneumoniae subsp. similipneumo...,ST1548,KL170,True,True,0
263,KSB1_6D_v2,Klebsiella quasipneumoniae subsp. similipneumo...,ST414,KL123,Klebsiella quasipneumoniae subsp. similipneumo...,ST414,KL123,True,True,0


In [27]:
# --- morphological variants -------------------------------------------------------------
# .1 and .2 are two colony morphologies off one isolate, so only one of each pair is kept.
pairs = survivors[survivors["isolate"].duplicated(keep=False)]
pairs[["sample", "isolate", "kbt_ont_ST", "kbt_ont_K_locus", "ext_ST", "ext_K_locus",
       "ext_mlst_distance", "asm_num_seqs", "asm_sum_len", "mapped_pct",
       "properly_paired_pct"]].sort_values(["isolate", "sample"])

,sample,isolate,kbt_ont_ST,kbt_ont_K_locus,ext_ST,ext_K_locus,ext_mlst_distance,asm_num_seqs,asm_sum_len,mapped_pct,properly_paired_pct
62,INF091.1,INF091,ST45,KL24,ST45,KL24,0,2,5508939,98.47,98.21
63,INF091.2,INF091,ST45,KL24,ST45,KL24,0,2,5600852,99.99,99.79
72,INF104.1,INF104,ST45,KL24,ST45,KL24,0,2,5499655,99.07,98.78
73,INF104.2,INF104,ST45,KL24,ST45,KL24,0,2,5556781,99.99,99.75
92,INF136.1,INF136,ST616,KL54,ST616,KL54,0,5,6020901,99.98,99.52
93,INF136.2,INF136,ST616,KL54,ST616,KL54,0,5,6042667,99.98,99.54
165,INF254.1,INF254,ST86,KL2,ST86,KL2,0,2,5452951,98.44,98.35
166,INF254.2,INF254,ST86,KL2,ST86,KL2,0,2,5452556,98.42,98.32
169,INF257.1_v2,INF257,ST3074,KL13,ST1536,KL30,4,2,5660285,99.62,99.24
170,INF257.2_v2,INF257,ST1536,KL30,ST1536,KL30,0,3,5352327,99.65,99.40


In [28]:
# Every pair has one closed chromosome, so completeness does not separate them. Rank on
# agreement with the reference first, then on how well the independent Illumina read set
# supports the assembly, then on contig count.
RANK = ["matches_reference", "mapped_pct", "properly_paired_pct", "fewest_contigs"]
survivors["fewest_contigs"] = -survivors["asm_num_seqs"]

ranked = survivors.sort_values(RANK + ["sample"], ascending=[False] * 4 + [True])
kept_variant = ranked.groupby("isolate", sort=False)["sample"].first()
survivors["variant_kept"] = survivors["sample"] == survivors["isolate"].map(kept_variant)

pairs = survivors[survivors["isolate"].duplicated(keep=False)]
pairs[["sample", "isolate", "matches_reference", "mapped_pct", "properly_paired_pct",
       "asm_num_seqs", "variant_kept"]].sort_values(["isolate", "sample"])

,sample,isolate,matches_reference,mapped_pct,properly_paired_pct,asm_num_seqs,variant_kept
62,INF091.1,INF091,True,98.47,98.21,2,False
63,INF091.2,INF091,True,99.99,99.79,2,True
72,INF104.1,INF104,True,99.07,98.78,2,False
73,INF104.2,INF104,True,99.99,99.75,2,True
92,INF136.1,INF136,True,99.98,99.52,5,False
93,INF136.2,INF136,True,99.98,99.54,5,True
165,INF254.1,INF254,True,98.44,98.35,2,True
166,INF254.2,INF254,True,98.42,98.32,2,False
169,INF257.1_v2,INF257,False,99.62,99.24,2,False
170,INF257.2_v2,INF257,True,99.65,99.40,3,True


In [29]:
# INF257 is decided on the first key: .2_v2 reproduces the reference, .1_v2 is a different
# strain (ST3074 / KL13, = reference isolate INF258) - two morphologies off that plate are
# two different strains, which is worth checking. INF315.1/.2 are the same assembly to 4 bp
# on every metric, so the tie falls to .1. The others are decided on mapped_pct.

In [30]:
# --- decisions --------------------------------------------------------------------------
# An assembly is kept if it agrees with itself; the reference then decides whether it is a
# plain pass or a flagged one. Dropped either as the losing variant of a pair, or because
# ONT and Illumina describe different strains - there is no assembly to keep in that case.
def decide(row):
    if not row["internally_concordant"]:
        return "DROP_INTERNAL_MISMATCH"
    if not row["variant_kept"]:
        return "DROP_VARIANT"
    if row["matches_reference"]:
        return "PASS"
    return "PASS_REFERENCE_MISMATCH"


survivors["external_qc_decision"] = survivors.apply(decide, axis=1)
survivors["external_qc_decision"].value_counts()

external_qc_decision
PASS                       279
PASS_REFERENCE_MISMATCH     15
DROP_VARIANT                 8
DROP_INTERNAL_MISMATCH       1
Name: count, dtype: int64

In [31]:
# Why, for the ones that are not a plain PASS
def mismatch_reason(row):
    if row["external_qc_decision"] == "DROP_INTERNAL_MISMATCH":
        return "ONT and Illumina libraries are different strains"
    if row["external_qc_decision"] != "PASS_REFERENCE_MISMATCH":
        return ""
    if row["ext_mlst_distance"] >= 3:
        return "reference row describes a different strain"
    if reference_K_locus_usable(row):
        return "K locus ambiguous on fragmented reference assembly"
    return "reference K locus call unusable"


survivors["mismatch_reason"] = survivors.apply(mismatch_reason, axis=1)
survivors[survivors["mismatch_reason"] != ""][
    ["sample", "external_qc_decision", "mismatch_reason", "best_external_match"]]

,sample,external_qc_decision,mismatch_reason,best_external_match
22,INF033,PASS_REFERENCE_MISMATCH,K locus ambiguous on fragmented reference asse...,
23,INF034_v2,PASS_REFERENCE_MISMATCH,reference row describes a different strain,"KSB1_7C,INF061,KSB1_5B"
109,INF167_v2,PASS_REFERENCE_MISMATCH,reference row describes a different strain,"INF300,INF359,KSB1_9J"
121,INF187_v2,PASS_REFERENCE_MISMATCH,reference row describes a different strain,"KSB1_10A,INF079"
131,INF202_v2,PASS_REFERENCE_MISMATCH,reference row describes a different strain,INF200
148,INF226_v2,PASS_REFERENCE_MISMATCH,reference row describes a different strain,INF224
149,INF227_v2,PASS_REFERENCE_MISMATCH,reference row describes a different strain,"INF191,INF225,INF254,INF331"
150,INF229_v2,PASS_REFERENCE_MISMATCH,reference row describes a different strain,INF226
162,INF251_v2,PASS_REFERENCE_MISMATCH,reference row describes a different strain,
167,INF255.1_v2,PASS_REFERENCE_MISMATCH,reference row describes a different strain,"INF261,INF282,INF289,INF314,INF246,INF332,INF1..."


In [32]:
# Dropping KSB2_5D_v2 costs the collection the isolate outright - nothing else represents
# KSB2_5D. Its ONT assembly is sound and stays in 06_asm_fixing; it is the pairing with a
# contaminated Illumina library that makes it unusable for a hybrid assembly.
survivors.loc[survivors["isolate"] == "KSB2_5D", ["sample", "external_qc_decision"]]

,sample,external_qc_decision
290,KSB2_5D_v2,DROP_INTERNAL_MISMATCH


In [33]:
# every kept assembly agrees with itself, and every isolate appears exactly once
kept = survivors[~survivors["external_qc_decision"].str.startswith("DROP")]
assert kept["internally_concordant"].all()
assert not kept["isolate"].duplicated().any()

len(kept), kept["isolate"].nunique()

(294, 294)

In [34]:
COLUMNS = ["sample", "isolate", "internal_qc_decision", "external_qc_decision",
           "mismatch_reason",
           "kbt_ont_species", "kbt_ont_ST", "kbt_ont_K_locus", "kbt_ont_K_locus_confidence",
           "ext_species", "ext_ST", "ext_K_locus", "ext_K_locus_confidence",
           "int_species_match", "int_K_locus_match", "int_mlst_distance",
           "ext_species_match", "ext_K_locus_match", "ext_mlst_distance",
           "internally_concordant", "matches_reference", "variant_kept",
           "best_external_match"]

summary = survivors[COLUMNS].sort_values("sample")
summary.to_csv("external_qc_summary.tsv", sep="\t", index=False)
summary

,sample,isolate,internal_qc_decision,external_qc_decision,mismatch_reason,kbt_ont_species,kbt_ont_ST,kbt_ont_K_locus,kbt_ont_K_locus_confidence,ext_species,ext_ST,ext_K_locus,ext_K_locus_confidence,int_species_match,int_K_locus_match,int_mlst_distance,ext_species_match,ext_K_locus_match,ext_mlst_distance,internally_concordant,matches_reference,variant_kept,best_external_match
0,INF001.1,INF001,PASS,PASS,,Klebsiella variicola subsp. variicola,ST3090-1LV,KL3,Typeable,Klebsiella variicola subsp. variicola,ST3090,KL3,Very high,True,True,0,True,True,0,True,True,True,
1,INF002,INF002,PASS,PASS,,Klebsiella pneumoniae,ST133,KL116,Typeable,Klebsiella pneumoniae,ST133,KL116,High,True,True,0,True,True,0,True,True,True,
2,INF003,INF003,PASS,PASS,,Klebsiella pneumoniae,ST323,KL21,Typeable,Klebsiella pneumoniae,ST323,KL21,Good,True,True,0,True,True,0,True,True,True,
3,INF004,INF004,PASS,PASS,,Klebsiella pneumoniae,ST345,KL15,Typeable,Klebsiella pneumoniae,ST345,KL15,Very high,True,True,0,True,True,0,True,True,True,
4,INF005,INF005,PASS,PASS,,Klebsiella variicola subsp. variicola,ST641,KL60,Typeable,Klebsiella variicola subsp. variicola,ST641,KL60,Very high,True,True,0,True,True,0,True,True,True,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297,KSB2_8A,KSB2_8A,PASS,PASS,,Klebsiella pneumoniae,ST463,KL117,Typeable,Klebsiella pneumoniae,ST463,KL117,Perfect,True,True,0,True,True,0,True,True,True,
298,KSB2_8C,KSB2_8C,PASS,PASS,,Klebsiella pneumoniae,ST2267,KL127,Typeable,Klebsiella pneumoniae,ST2267,KL127,Very high,True,True,0,True,True,0,True,True,True,
299,KSB2_8D,KSB2_8D,PASS,PASS,,Klebsiella pneumoniae,ST643,KL38,Typeable,Klebsiella pneumoniae,ST643,KL38,Very high,True,True,0,True,True,0,True,True,True,
300,KSB2_9B,KSB2_9B,PASS,PASS,,Klebsiella pneumoniae,ST25,KL2,Typeable,Klebsiella pneumoniae,ST25,KL2,Very high,True,True,0,True,True,0,True,True,True,


In [35]:
kept["external_qc_decision"].value_counts()

external_qc_decision
PASS                       279
PASS_REFERENCE_MISMATCH     15
Name: count, dtype: int64

In [36]:
# --- sanity check -----------------------------------------------------------------------
# The typing above is all Kleborate. The assemblies themselves have not been looked at, and
# the six recovered ones still carry their pre-fix asm_* columns in internal_qc_summary,
# so measure their FASTAs rather than trusting those.
CHROMOSOME_MIN_LEN = 4_000_000          # bin/scripts/integrate_qc.py definition
KPSC_CHROMOSOME_SIZE = (4_800_000, 6_200_000)

FIXED_ASSEMBLY = {
    "INF205": "../06_asm_fixing/01_autocycler/assemblies/INF205.fasta",
    **{strain: f"../06_asm_fixing/02_decontam/assemblies/{strain}.fasta"
       for strain in ["INF186", "INF216", "KSB1_2I", "KSB2_4A", "KSB2_5D_v2"]},
}
asm = qc.set_index("sample")


def contig_lengths(path):
    lengths, current = [], 0
    for line in open(path):
        if line.startswith(">"):
            lengths.append(current)
            current = 0
        else:
            current += len(line.strip())
    return [length for length in lengths[1:] + [current] if length]


def chromosomes(sample):
    if sample in FIXED_ASSEMBLY:
        return [length for length in contig_lengths(FIXED_ASSEMBLY[sample])
                if length >= CHROMOSOME_MIN_LEN]
    return [asm.loc[sample, "asm_max_len"]] * int(asm.loc[sample, "asm_n_chromosomes"])


chromosome = kept["sample"].map(chromosomes)
kept = kept.assign(n_chromosomes=chromosome.map(len),
                   chromosome_len=chromosome.map(max))

kept[["n_chromosomes", "chromosome_len"]].agg(["min", "max"])

,n_chromosomes,chromosome_len
min,1,5031175
max,1,5660325


In [37]:
# Every kept assembly holds exactly one chromosome, of a size credible for KpSC
assert (kept["n_chromosomes"] == 1).all()
assert kept["chromosome_len"].between(*KPSC_CHROMOSOME_SIZE).all()

# and the two decisions partition the 294 exactly as the reference comparison says
assert (kept["external_qc_decision"] == "PASS").equals(kept["matches_reference"])
assert (kept["external_qc_decision"] == "PASS_REFERENCE_MISMATCH").equals(
    ~kept["matches_reference"])

len(kept)

294

In [38]:
# 294 assemblies, one per isolate, each internally concordant and each with a single
# chromosome. 279 reproduce the reference; 15 are sound but disagree with their metadata row.
kept.groupby(["external_qc_decision", "mismatch_reason"]).size()

external_qc_decision     mismatch_reason                                   
PASS                                                                           279
PASS_REFERENCE_MISMATCH  K locus ambiguous on fragmented reference assembly      2
                         reference K locus call unusable                         2
                         reference row describes a different strain             11
dtype: int64